# Lab 2 · Clean, connect and check

**In class:** Part A, guided, on a messy version of the penguins data · **At home:** Part B, the same steps on your own two tables
**Due:** Sunday, November 8 at 11:59 PM (extended for Election Day) · **Submit:** your repo URL on CourseWorks

Tonight's three skills:
1. **Clean** a messy CSV with pandas and load it into DuckDB
2. **Connect** two tables with a join, on a key you've checked
3. **Check** every join by counting rows before and after

Run cells top to bottom with **Shift+Enter**.

---
# Part A · Guided: messy penguins + a lookup table

## A1. Read the messy file with pandas

`penguins_raw.csv` is the same study as last week's penguins, in its original form: long column names, extra fields, comments. **pandas** is a Python library for working with tables; a table in pandas is called a **DataFrame**.

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / "data" / "penguins_raw.csv")

print(raw.shape)   # (rows, columns)
raw.head()

(344, 17)


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


How much is missing in each column? `isna()` marks missing values; `.sum()` counts them per column.

In [4]:
raw.isna().sum()

studyName                0
Sample Number            0
Species                  0
Region                   0
Island                   0
Stage                    0
Individual ID            0
Clutch Completion        0
Date Egg                 0
Culmen Length (mm)       2
Culmen Depth (mm)        2
Flipper Length (mm)      2
Body Mass (g)            2
Sex                     11
Delta 15 N (o/oo)       14
Delta 13 C (o/oo)       13
Comments               290
dtype: int64

Look at the species column. What would make it hard to use as a key?

In [5]:
raw["Species"].value_counts()

Species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64

**Notice:** long names with the Latin name in parentheses, and inconsistent capitals ("Adelie **P**enguin" vs. "Gentoo **p**enguin"). Text that *looks* the same to a person often won't match exactly in a join.

## A2. Three cleaning steps

**Step 1. Rename columns** to short, lowercase names with no spaces, so they're easy to type in SQL.

In [6]:
penguins = raw.rename(columns={
    "studyName": "study",
    "Individual ID": "individual_id",
    "Species": "species_raw",
    "Island": "island",
    "Date Egg": "date_egg",
    "Clutch Completion": "clutch_completion",
    "Culmen Length (mm)": "bill_length_mm",
    "Culmen Depth (mm)": "bill_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
})
penguins.columns.tolist()

['study',
 'Sample Number',
 'species_raw',
 'Region',
 'island',
 'Stage',
 'individual_id',
 'clutch_completion',
 'date_egg',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex',
 'Delta 15 N (o/oo)',
 'Delta 13 C (o/oo)',
 'Comments']

**Step 2. Standardize text.** Make `sex` lowercase, and pull the Latin name out of the parentheses into its own column. Scientific names are written the same way everywhere, which makes them a better key than the common name.

In [7]:
penguins["sex"] = penguins["sex"].str.lower()
penguins["latin_name"] = penguins["species_raw"].str.extract(r"\((.*)\)")   # text inside ( )

penguins[["species_raw", "latin_name", "sex"]].drop_duplicates().head(6)

,species_raw,latin_name,sex
0,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,male
1,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,female
3,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,NaN
152,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,female
153,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,male
178,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,NaN


**Step 3. Fix types and keep what you need.** The egg date came in as text; make it a real date. Then keep only the columns we'll use.

In [8]:
penguins["date_egg"] = pd.to_datetime(penguins["date_egg"])

penguins = penguins[[
    "study", "individual_id", "latin_name", "island", "date_egg", "clutch_completion",
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex",
]]
penguins.dtypes

study                           str
individual_id                   str
latin_name                      str
island                          str
date_egg             datetime64[us]
clutch_completion               str
bill_length_mm              float64
bill_depth_mm               float64
flipper_length_mm           float64
body_mass_g                 float64
sex                             str
dtype: object

## A3. Load it into DuckDB and verify with SQL

DuckDB can read a pandas DataFrame directly by its variable name. Always check the row count after moving data: **rows in should equal rows out.**

In [8]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

print("rows in pandas:", len(penguins))
con.sql("SELECT count(*) AS rows_in_duckdb FROM penguins_clean")

CatalogException: Catalog Error: Table with name penguins does not exist!
Did you mean "penguins_clean"?

LINE 1: CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins
                                                                ^

In [10]:
con.sql("DESCRIBE penguins_clean")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ study             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ individual_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latin_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_egg          │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ clutch_completion │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_

## A4. What is one row? Is the ID really an ID?

There's a column called `individual_id`. If it identified one penguin, every penguin would appear once per season. Let's check.

In [11]:
con.sql("""
    SELECT count(*)                      AS rows,
           count(DISTINCT individual_id) AS distinct_ids
    FROM penguins_clean
""")

┌───────┬──────────────┐
│ rows  │ distinct_ids │
│ int64 │    int64     │
├───────┼──────────────┤
│   344 │          190 │
└───────┴──────────────┘

In [12]:
con.sql("""
    SELECT study, latin_name, island, individual_id
    FROM penguins_clean
    WHERE individual_id = 'N11A1'
""")

┌─────────┬────────────────────┬─────────┬───────────────┐
│  study  │     latin_name     │ island  │ individual_id │
│ varchar │      varchar       │ varchar │    varchar    │
├─────────┼────────────────────┼─────────┼───────────────┤
│ PAL0708 │ Pygoscelis adeliae │ Biscoe  │ N11A1         │
│ PAL0809 │ Pygoscelis papua   │ Biscoe  │ N11A1         │
└─────────┴────────────────────┴─────────┴───────────────┘

**The same ID belongs to an Adélie penguin in one season and a Gentoo the next.** It can't be one bird. It's most likely a nest-and-adult label the researchers reused. So:

- `individual_id` **alone is not a key.** A column named "ID" isn't automatically unique; always check with `count(DISTINCT ...)`.
- **One row = one adult penguin at one nest in one breeding season.** That's the table's **grain**, and you should be able to say it for every table you use.

## A5. A lookup table to join

`species_lookup.csv` is a small reference table: one row per species, with its common name, Latin name, genus and group.

In [13]:
con.sql(f"CREATE OR REPLACE TABLE species AS SELECT * FROM read_csv('{ROOT / 'data' / 'species_lookup.csv'}')")
con.sql("SELECT * FROM species")

┌───────────────────┬────────────────────────┬─────────────┬───────────────────────┐
│    common_name    │       latin_name       │    genus    │      group_name       │
│      varchar      │        varchar         │   varchar   │        varchar        │
├───────────────────┼────────────────────────┼─────────────┼───────────────────────┤
│ Adélie penguin    │ Pygoscelis adeliae     │ Pygoscelis  │ brush-tailed penguins │
│ Chinstrap penguin │ Pygoscelis antarcticus │ Pygoscelis  │ brush-tailed penguins │
│ Gentoo penguin    │ Pygoscelis papua       │ Pygoscelis  │ brush-tailed penguins │
│ Emperor penguin   │ Aptenodytes forsteri   │ Aptenodytes │ great penguins        │
└───────────────────┴────────────────────────┴─────────────┴───────────────────────┘

Before joining, check that the key is **unique** in the lookup table. If a key appeared twice, every matching penguin would be copied twice.

In [14]:
con.sql("""
    SELECT latin_name, count(*) AS n
    FROM species
    GROUP BY latin_name
    HAVING count(*) > 1
""")   # no rows back = the key is unique

┌────────────┬───────┐
│ latin_name │   n   │
│  varchar   │ int64 │
└────────────┴───────┘
        0 rows      

## A6. Join, and count what comes out

`JOIN ... ON` lines up rows where the key matches. We expect **344** rows out, one per penguin row.

In [15]:
con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             276 │
└─────────────────┘

**Only 276, not 344.** An inner `JOIN` silently drops rows that don't match. Nothing errored. This is a data cascade in the making.

## A7. Find what didn't match with a LEFT JOIN

A `LEFT JOIN` keeps every penguin row. Where there's no match, the lookup's columns come back empty (`NULL`), so we can find the problem rows.

In [16]:
con.sql("""
    SELECT p.latin_name, count(*) AS unmatched_rows
    FROM penguins_clean p
    LEFT JOIN species s ON p.latin_name = s.latin_name
    WHERE s.latin_name IS NULL
    GROUP BY p.latin_name
""")

┌───────────────────────┬────────────────┐
│      latin_name       │ unmatched_rows │
│        varchar        │     int64      │
├───────────────────────┼────────────────┤
│ Pygoscelis antarctica │             68 │
└───────────────────────┴────────────────┘

Compare with the lookup: the data says **Pygoscelis antarctic*a***, the lookup says **Pygoscelis antarctic*us***. Two sources spelled the same species differently, and 68 penguins vanished from the join.

## A8. Fix the key, then join again

Fix it at the source of the problem (the cleaning step), then reload and rejoin.

In [17]:
penguins["latin_name"] = penguins["latin_name"].replace(
    {"Pygoscelis antarctica": "Pygoscelis antarcticus"}
)
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")   # now 344

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             344 │
└─────────────────┘

Which way you join matters. Start from the **lookup** with a `LEFT JOIN` to see species that have no penguins:

In [18]:
con.sql("""
    SELECT s.common_name, count(p.latin_name) AS penguin_rows
    FROM species s
    LEFT JOIN penguins_clean p ON p.latin_name = s.latin_name
    GROUP BY s.common_name
    ORDER BY penguin_rows DESC
""")

┌───────────────────┬──────────────┐
│    common_name    │ penguin_rows │
│      varchar      │    int64     │
├───────────────────┼──────────────┤
│ Adélie penguin    │          152 │
│ Gentoo penguin    │          124 │
│ Chinstrap penguin │           68 │
│ Emperor penguin   │            0 │
└───────────────────┴──────────────┘

## A9. What a duplicate key does

Here we fake a lookup table that lists Gentoo twice, then join. Watch the row count.

In [19]:
con.sql("""
    WITH species_with_dupe AS (
        SELECT * FROM species
        UNION ALL
        SELECT * FROM species WHERE common_name = 'Gentoo penguin'
    )
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species_with_dupe s ON p.latin_name = s.latin_name
""")   # 468: every Gentoo row was copied

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             468 │
└─────────────────┘

**344 became 468**, and every total or average involving Gentoos would now be wrong. That's why you check key uniqueness (A5) before every join.

## A10. Use the join to answer a question

Now that the tables connect, ask a question that needs both: the lookup's readable names, the penguin table's measurements.

In [20]:
con.sql("""
    SELECT s.common_name,
           p.island,
           count(*)                    AS penguins,
           round(avg(p.body_mass_g))   AS avg_mass_g
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
    GROUP BY s.common_name, p.island
    ORDER BY s.common_name, p.island
""")

┌───────────────────┬───────────┬──────────┬────────────┐
│    common_name    │  island   │ penguins │ avg_mass_g │
│      varchar      │  varchar  │  int64   │   double   │
├───────────────────┼───────────┼──────────┼────────────┤
│ Adélie penguin    │ Biscoe    │       44 │     3710.0 │
│ Adélie penguin    │ Dream     │       56 │     3688.0 │
│ Adélie penguin    │ Torgersen │       52 │     3706.0 │
│ Chinstrap penguin │ Dream     │       68 │     3733.0 │
│ Gentoo penguin    │ Biscoe    │      124 │     5076.0 │
└───────────────────┴───────────┴──────────┴────────────┘

### The join checklist (use it in Part B)

1. **Grain:** what is one row in each table?
2. **Key:** which column connects them, and is it **unique** in the lookup side?
3. **Clean the key** the same way on both sides (spelling, capitals, spaces, codes vs. names, types).
4. **Count rows** before and after. More rows = a duplicate key. Fewer = unmatched keys.
5. **LEFT JOIN … IS NULL** to see exactly what didn't match, then decide: fix, drop, or keep as unknown.

---
# Part B · At home: your own two tables

Apply the checklist to the two tables you chose in Lab 1. If you'd like to swap datasets, that's fine; you just need two public tables that share a column.

## B1. Load both tables with pandas

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

In [2]:
print("Kernel OK")
print("DuckDB version:", duckdb.__version__)

Kernel OK
DuckDB version: 1.5.6


In [3]:
con = duckdb.connect("lab2.duckdb")

In [5]:
con.execute("SET memory_limit = '1GB'")
con.execute("SET threads = 1")

In [8]:
FILE_A = "POPS.csv"     # <- your main table's file in data/
FILE_B = "PLUTO.csv"   # <- the related table's file in data/

a = pd.read_csv(ROOT / "data" / FILE_A)

pluto_cols = [
    "BBL", "address", "landuse", "bldgclass",
    "lotarea", "bldgarea", "comarea", "resarea",
    "officearea", "retailarea", "numfloors",
    "unitsres", "yearbuilt", "zonedist1",
    "ownertype"
]

b = pd.read_csv(
    ROOT / "data" / FILE_B,
    usecols=pluto_cols,
)

print("A:", a.shape, "  B:", b.shape)
a.head()

/tmp/ipykernel_47288/1509565761.py:14: DtypeWarning: Columns (44: unitsres) have mixed types. Specify dtype option on import or set low_memory=False.
  b = pd.read_csv(


A: (392, 34)   B: (858284, 15)


,POPS_Number,Borough,Borocode,Community Board,Number,Street,Postcode,Building_Address_With_Zip_Code,Tax_Block,Tax_Lot,...,Latitude,Longitude,XCoordinate,YCoordinate,Council District,Census Tract,BIN,BBL,NTA,Location
0,M050022,Manhattan,1,5,1350,SIXTH AVENUE,10019,"1350 SIXTH AVENUE, New York, NY 10019","1,270",71.0,...,40.762575,-73.977778,"990,406","217,104",4,104.00,1034815.0,1.012700e+09,NaN,POINT (-73.977778 40.762575)
1,M050065,Manhattan,1,5,3,PARK AVENUE,10016,"3 PARK AVENUE, New York, NY 10016",889,7501.0,...,40.746511,-73.981075,"989,494","211,251",4,72.00,1087537.0,1.008898e+09,NaN,POINT (-73.981075 40.746511)
2,M050117,Manhattan,1,5,415,MADISON AVENUE,10017,"415 MADISON AVENUE, New York, NY 10017","1,284",21.0,...,40.756662,-73.976058,"990,883","214,950",4,94.00,1000000.0,1.012840e+09,NaN,POINT (-73.976058 40.756662)
3,M080022,Manhattan,1,8,353,EAST 72 STREET,10021,"353 EAST 72 STREET, New York, NY 10021","1,447",19.0,...,40.768300,-73.956462,"996,310","219,192",5,126.00,1044882.0,1.014470e+09,NaN,POINT (-73.956462 40.7683)
4,M080069,Manhattan,1,8,1675,YORK AVENUE,10128,"1675 YORK AVENUE, New York, NY 10128","1,568",21.0,...,40.777710,-73.946331,"999,114","222,622",5,144.02,1050621.0,1.015680e+09,NaN,POINT (-73.946331 40.77771)


In [9]:
b.head()

,address,zonedist1,bldgclass,landuse,ownertype,lotarea,bldgarea,comarea,resarea,officearea,retailarea,numfloors,unitsres,yearbuilt,BBL
0,84-74 257 STREET,R2A,A1,1.0,NaN,"4,000","1,864",0,"1,864",0,0,2.0,1,1945.0,4087860042
1,84-82 257 STREET,R2A,A0,1.0,NaN,"3,100",960,0,960,0,0,1.5,1,1945.0,4087860046
2,85-01 256 STREET,R2A,A0,1.0,NaN,"4,000","1,068",0,"1,068",0,0,1.5,1,1945.0,4087860051
3,85-59 LITTLE NECK PARKWAY,R2A,A0,1.0,NaN,"4,794",960,0,960,0,0,1.5,1,1945.0,4087980002
4,85-55 LITTLE NECK PARKWAY,R2A,A0,1.0,NaN,"4,520",960,0,960,0,0,1.5,1,1945.0,4087980003


In [10]:
b["BBL"].dtypes
a["BBL"].dtypes

dtype('float64')

**Grain** (double-click to edit). One row in table A is: A POPS. One row in table B is: A tax lot.

## B2. Clean: at least three steps

Use the Part A steps as a model: rename columns, standardize text, fix types, drop columns you don't need. Write each step in its own line with a comment saying *why*.

In [11]:
# Step 1: rename columns (short, lowercase, no spaces)
# a = a.rename(columns={"Country Code": "country_code", ...})

# Step 2: standardize the key text so both tables match
#a["BBL"] = a["BBL"].str.strip().str.upper()
#b["BBL"] = b["BBL"].str.strip().str.upper()

a["BBL"] = a["BBL"].astype("Int64")
b["BBL"] = b["BBL"].astype("Int64")

# Step 3: fix a type, or drop rows/columns you don't need
# a["year"] = a["year"].astype(int)

a.head()

,POPS_Number,Borough,Borocode,Community Board,Number,Street,Postcode,Building_Address_With_Zip_Code,Tax_Block,Tax_Lot,...,Latitude,Longitude,XCoordinate,YCoordinate,Council District,Census Tract,BIN,BBL,NTA,Location
0,M050022,Manhattan,1,5,1350,SIXTH AVENUE,10019,"1350 SIXTH AVENUE, New York, NY 10019","1,270",71.0,...,40.762575,-73.977778,"990,406","217,104",4,104.00,1034815.0,1012700071,NaN,POINT (-73.977778 40.762575)
1,M050065,Manhattan,1,5,3,PARK AVENUE,10016,"3 PARK AVENUE, New York, NY 10016",889,7501.0,...,40.746511,-73.981075,"989,494","211,251",4,72.00,1087537.0,1008897501,NaN,POINT (-73.981075 40.746511)
2,M050117,Manhattan,1,5,415,MADISON AVENUE,10017,"415 MADISON AVENUE, New York, NY 10017","1,284",21.0,...,40.756662,-73.976058,"990,883","214,950",4,94.00,1000000.0,1012840021,NaN,POINT (-73.976058 40.756662)
3,M080022,Manhattan,1,8,353,EAST 72 STREET,10021,"353 EAST 72 STREET, New York, NY 10021","1,447",19.0,...,40.768300,-73.956462,"996,310","219,192",5,126.00,1044882.0,1014470019,NaN,POINT (-73.956462 40.7683)
4,M080069,Manhattan,1,8,1675,YORK AVENUE,10128,"1675 YORK AVENUE, New York, NY 10128","1,568",21.0,...,40.777710,-73.946331,"999,114","222,622",5,144.02,1050621.0,1015680021,NaN,POINT (-73.946331 40.77771)


## B3. Load into DuckDB and check the key

In [12]:
import duckdb
con = duckdb.connect()

con.sql("SELECT * FROM a")
con.sql("SELECT * FROM b")

KEY_A = "BBL"   # <- after your renames
KEY_B = "BBL"

# Is the key unique in table B? No rows back = unique.
con.sql(f"""
    SELECT "{KEY_B}", count(*) AS n
    FROM b
    GROUP BY "{KEY_B}"
    HAVING count(*) > 1
    LIMIT 10
""")

┌───────┬───────┐
│  BBL  │   n   │
│ int64 │ int64 │
└───────┴───────┘
     0 rows    

## B4. Join and count

In [13]:
print("rows in table A:", con.sql("SELECT count(*) FROM a").fetchone()[0])

con.sql(f"""
    SELECT count(*) AS rows_after_join
    FROM a a
    JOIN b b ON a."{KEY_A}" = b."{KEY_B}"
""")

rows in table A: 392


┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             385 │
└─────────────────┘

## B5. Find what didn't match

In [14]:
con.sql(f"""
    SELECT a."{KEY_A}", count(*) AS unmatched_rows
    FROM a a
    LEFT JOIN b b ON a."{KEY_A}" = b."{KEY_B}"
    WHERE b."{KEY_B}" IS NULL
    GROUP BY a."{KEY_A}"
    ORDER BY unmatched_rows DESC
    LIMIT 20
""")

┌────────────┬────────────────┐
│    BBL     │ unmatched_rows │
│   int64    │     int64      │
├────────────┼────────────────┤
│       NULL │              2 │
│ 1007297503 │              1 │
│ 1000380001 │              1 │
│ 1015630004 │              1 │
│ 1010090019 │              1 │
│ 1011710157 │              1 │
└────────────┴────────────────┘

In [15]:
con.sql("""
    SELECT BBL
    FROM b
    WHERE BBL IN (
        1007297503, 1015630004, 1000380001,
        1010090019, 1011710157
    )
""")

┌────────┐
│  BBL   │
│ int64  │
└────────┘
  0 rows

In [16]:
con.sql("DESCRIBE b")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ address     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ zonedist1   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bldgclass   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ landuse     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ownertype   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ lotarea     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bldgarea    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ comarea     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ resarea     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ officearea  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ retailarea  │ VARC

In [17]:
query = """
WITH unmatched AS (
    -- Find POPS records without a PLUTO BBL match
    SELECT
        p.BBL AS pops_bbl,
        p.Building_Address_With_Zip_Code AS pops_address
    FROM a p
    LEFT JOIN b l
        ON TRY_CAST(p.BBL AS BIGINT) =
           TRY_CAST(l.BBL AS BIGINT)
    WHERE p.BBL IS NOT NULL
      AND l.BBL IS NULL
)

-- Find alternative matches using addresses
SELECT
    u.pops_bbl,
    u.pops_address,
    l.BBL AS candidate_pluto_bbl,
    l.Address AS pluto_address,
    l.LandUse,
    l.BldgClass
FROM unmatched u
LEFT JOIN b l
    ON UPPER(TRIM(SPLIT_PART(u.pops_address, ',', 1)))
       = UPPER(TRIM(l.Address))
ORDER BY u.pops_bbl
"""

display(con.execute(query).df())

,pops_bbl,pops_address,candidate_pluto_bbl,pluto_address,landuse,bldgclass
0,1000380001,"100 WALL STREET, New York, NY 10005",1000387501,100 WALL STREET,NaN,R0
1,1000380001,"100 WALL STREET, New York, NY 10005",1000381003,100 WALL STREET,5.0,RB
2,1000380001,"100 WALL STREET, New York, NY 10005",1000381001,100 WALL STREET,5.0,RK
3,1007297503,"450 WEST 33 STREET, New York, NY 10001",<NA>,NaN,NaN,NaN
4,1010090019,"118 WEST 57 STREET, New York, NY 10019",<NA>,NaN,NaN,NaN
5,1011710157,"639 WEST 59 STREET, New York, NY 10019",<NA>,NaN,NaN,NaN
6,1015630004,"400 EAST 84 STREET, New York, NY 10028",<NA>,NaN,NaN,NaN


In [18]:
# Find PLUTO records in the same tax blocks
con.sql("""
    SELECT
        BBL,
        address,
        landuse,
        bldgclass
    FROM b
    WHERE FLOOR(BBL / 10000) IN (
        100729,
        101009,
        101171,
        101563
    )
    ORDER BY BBL
""").df()

,BBL,address,landuse,bldgclass
0,1007290001,WEST 31 STREET,7.0,U1
1,1007290006,NaN,7.0,U9
2,1007290015,WEST 31 STREET,7.0,U1
3,1007290050,9 AVENUE,7.0,U6
4,1007290163,424 WEST 33 STREET,5.0,O6
...,...,...,...,...
122,1015630129,452 EAST 84 STREET,2.0,C4
123,1015630139,428 EAST 84 STREET,1.0,B1
124,1015630141,422 EAST 84 STREET,1.0,A9
125,1015630144,414 EAST 84 STREET,2.0,C1


In [19]:
# Find candidate lots within the same tax block
con.sql("""
    SELECT BBL, address, landuse, bldgclass
    FROM b
    WHERE
        (FLOOR(BBL / 10000) = 100729
         AND address LIKE '%WEST 33%')
    OR
        (FLOOR(BBL / 10000) = 101009
         AND address LIKE '%WEST 57%')
    OR
        (FLOOR(BBL / 10000) = 101171
         AND address LIKE '%WEST 59%')
    ORDER BY BBL
""").df()

,BBL,address,landuse,bldgclass
0,1007290163,424 WEST 33 STREET,5.0,O6
1,1007297506,438 WEST 33RD STREET,5.0,RC
2,1007297507,450 WEST 33RD STREET,5.0,RC
3,1010090005,152 WEST 57 STREET,5.0,O4
4,1010090009,150 WEST 57 STREET,5.0,K2
5,1010090037,102 WEST 57 STREET,5.0,H3
6,1010090040,110 WEST 57 STREET,5.0,O6
7,1010090044,120 WEST 57 STREET,5.0,H2
8,1010090046,130 WEST 57 STREET,5.0,O6
9,1010090050,140 WEST 57 STREET,5.0,O6


**What didn't match, and what will you do about it?** (fix the key, drop those rows, or keep them as unknown, and why) 
Seven POPS records did not match PLUTO by BBL. Two have missing BBLs, while five may reflect differences in parcel identifiers, including condominium lot numbers. I will keep these records as unmatched for now and verify them using other datasets before assigning alternative BBLs.



## B6. Answer one question that needs both tables

Write a query that joins the tables and uses columns from **both**, e.g. average GDP by region, complaints per 1,000 residents by ZIP code.

In [20]:
# Count POPS by their associated building land-use category
con.sql("""
    SELECT
        b.landuse,
        COUNT(*) AS pops_count
    FROM a
    JOIN b
        ON a.BBL = b.BBL
    GROUP BY b.landuse
    ORDER BY pops_count DESC
""")

┌─────────┬────────────┐
│ landuse │ pops_count │
│ double  │   int64    │
├─────────┼────────────┤
│     5.0 │        154 │
│     4.0 │        144 │
│     3.0 │         67 │
│     8.0 │         14 │
│    10.0 │          3 │
│     7.0 │          1 │
│     9.0 │          1 │
│    11.0 │          1 │
└─────────┴────────────┘

**What does the result tell you?** (one or two sentences) Among the 385 matched POPS records, commercial and office buildings account for the largest share, followed by mixed residential and commercial buildings and multi-family elevator buildings. Together, these three land-use categories account for approximately more than 90% of matched POPS records.

## B7. Submit

1. Run the cell below.
2. Answer the two reflection questions in `labs/lab2.md`.
3. Save this notebook (Cmd+S), then commit and push: **Source Control** → `Lab 2` → **Commit** → **Sync Changes**.
4. Paste your repo URL into the Lab 2 assignment on CourseWorks.

In [21]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
